# 04 — Model Development

LightGBM demand forecasting with Optuna tuning, walk-forward evaluation, SHAP analysis, and error breakdown by segment.

Run `python ml/train.py` first to generate the trained model artifacts.

In [ ]:
import sys
sys.path.insert(0, '../..')

import io, json, pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from ml.athena_client import get_s3_client, BUCKET
from ml.train import (
    FEATURE_COLS, CATEGORICAL_COLS, MODEL_VERSION, MODEL_S3_PREFIX,
    FEATURES_S3_KEY, FOLDS, wape, rmse, bias, fva, make_lgb_dataset
)

s3 = get_s3_client()

# load features
obj = s3.get_object(Bucket=BUCKET, Key=FEATURES_S3_KEY)
df  = pd.read_parquet(io.BytesIO(obj['Body'].read()))
df['date'] = pd.to_datetime(df['date'])
for col in FEATURE_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
day0 = df['date'].min()

# load trained models
obj    = s3.get_object(Bucket=BUCKET, Key=f'{MODEL_S3_PREFIX}/demand_forecast_lgbm_{MODEL_VERSION}.pkl')
models = pickle.loads(obj['Body'].read())

# load metadata
obj      = s3.get_object(Bucket=BUCKET, Key=f'{MODEL_S3_PREFIX}/demand_forecast_lgbm_{MODEL_VERSION}_metadata.json')
metadata = json.loads(obj['Body'].read())

print(f'Features: {len(df):,} rows')
print(f'Models loaded for horizons: {list(models.keys())}')
print(f'Best Optuna WAPE: {metadata["optuna_best_wape"]:.4f}')

## 1. Walk-forward CV results

In [ ]:
from ml.evaluate import full_walkforward_eval

wf = full_walkforward_eval(models, df, day0)

print('Walk-forward results by fold and horizon:')
pivot = wf.pivot_table(index='horizon', columns='fold', values='wape').round(4)
print(pivot)

print(f'\nOverall mean WAPE: {wf["wape"].mean():.4f}')
print(f'Seasonal naive WAPE: {metadata["baseline_metrics"]["seasonal_naive"]["wape"]:.4f}')
print(f'FVA: {fva(wf["wape"].mean(), metadata["baseline_metrics"]["seasonal_naive"]["wape"]):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ['wape', 'rmse', 'bias']):
    wf.groupby('horizon')[metric].mean().plot(ax=ax, marker='o', color='steelblue')
    ax.set_title(f'{metric.upper()} by horizon (mean across folds)')
    ax.set_xlabel('Horizon (days ahead)')
    ax.axhline(0, color='gray', lw=0.5, ls='--')

plt.tight_layout()
plt.show()

## 2. Segment error analysis

In [ ]:
from ml.evaluate import segment_error_analysis

segments = segment_error_analysis(models, df, day0)

for seg_name, seg_data in segments.items():
    seg_df = pd.DataFrame(seg_data).T.sort_values('wape', ascending=False)
    print(f'\n── By {seg_name} ──')
    print(seg_df[['wape', 'rmse', 'bias', 'n']].round(4).to_string())

In [ ]:
# visualise category breakdown
cat_data = pd.DataFrame(segments.get('category', {})).T
if not cat_data.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    cat_data['wape'].astype(float).sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.axvline(metadata['baseline_metrics']['seasonal_naive']['wape'],
               color='red', ls='--', label='Seasonal naive WAPE')
    ax.set_title('WAPE by product category')
    ax.set_xlabel('WAPE')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. SHAP analysis

In [ ]:
from ml.evaluate import run_shap_analysis

shap_results = run_shap_analysis(models, df, day0)

print('Global feature importance (mean |SHAP|):')
for feat, val in list(shap_results['global_importance'].items())[:15]:
    bar = '█' * int(val / max(shap_results['global_importance'].values()) * 30)
    print(f'  {feat:30s} {bar} {val:.4f}')

print(f'\nTop feature: {shap_results["top_feature"]}')
print(f'Lag vs calendar SHAP ratio: {shap_results["lag_vs_calendar_ratio"]:.2f}')
print('  (>1 means lag features dominate; <1 means calendar features dominate)')

print('\nTop-5 features on worst-predicted pairs:')
for feat, val in shap_results['worst_pairs_importance'].items():
    print(f'  {feat:30s} {val:.4f}')

In [ ]:
# SHAP beeswarm on h=1 model
fold = FOLDS[-1]
train_end = day0 + pd.Timedelta(days=fold['train_end'] - 1)
val_start = day0 + pd.Timedelta(days=fold['val_start'] - 1)
val_end   = day0 + pd.Timedelta(days=fold['val_end']   - 1)

train_df = df[df['date'] <= train_end]
val_df   = df[(df['date'] >= val_start) & (df['date'] <= val_end)]

X_tr, y_tr   = make_lgb_dataset(train_df, 1)
X_val, y_val = make_lgb_dataset(val_df, 1)

m = lgb.LGBMRegressor(
    **{k: v for k, v in models[1].get_params().items() if k != 'n_estimators'},
    n_estimators=500, random_state=42, verbose=-1
)
m.fit(X_tr, y_tr)

sample = X_val[FEATURE_COLS].sample(min(300, len(X_val)), random_state=42)
explainer   = shap.TreeExplainer(m)
shap_values = explainer.shap_values(sample)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, sample, plot_type='dot', show=False, max_display=20)
plt.title('SHAP beeswarm — h=1 model (fold 4 validation sample)')
plt.tight_layout()
plt.show()

## 4. Model vs baseline comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Naive', 'Seasonal naive', 'Rolling mean 7d', 'LightGBM (mean h=1-7)'],
    'WAPE':  [
        metadata['baseline_metrics']['naive']['wape'],
        metadata['baseline_metrics']['seasonal_naive']['wape'],
        metadata['baseline_metrics']['rolling_mean']['wape'],
        wf['wape'].mean(),
    ],
    'RMSE':  [
        metadata['baseline_metrics']['naive']['rmse'],
        metadata['baseline_metrics']['seasonal_naive']['rmse'],
        metadata['baseline_metrics']['rolling_mean']['rmse'],
        wf['rmse'].mean(),
    ],
    'Bias':  [
        metadata['baseline_metrics']['naive']['bias'],
        metadata['baseline_metrics']['seasonal_naive']['bias'],
        metadata['baseline_metrics']['rolling_mean']['bias'],
        wf['bias'].mean(),
    ],
})
sn_wape = metadata['baseline_metrics']['seasonal_naive']['wape']
comparison['FVA'] = comparison['WAPE'] / sn_wape

print('Model comparison:')
print(comparison.round(4).to_string(index=False))
print()
lgbm_fva = comparison[comparison['Model'].str.contains('LightGBM')]['FVA'].values[0]
if lgbm_fva < 1.0:
    print(f'✓ LightGBM FVA = {lgbm_fva:.3f} < 1.0 — model beats seasonal naive. DEPLOY.')
else:
    print(f'✗ LightGBM FVA = {lgbm_fva:.3f} ≥ 1.0 — model does NOT beat seasonal naive. DO NOT DEPLOY.')